# 🔍 Modificando e Consultando Valores em DataFrame

![Jupyter](https://img.shields.io/badge/Jupyter-111827?style=flat-square&logo=jupyter&logoColor=F37626)
![Python](https://img.shields.io/badge/Python-111827?style=flat-square&logo=python&logoColor=3776AB)
![Pandas](https://img.shields.io/badge/Pandas-150458?style=flat-square&logo=pandas&logoColor=white)
![Python Version](https://img.shields.io/badge/python-3.14+-blue)
![Tópico](https://img.shields.io/badge/tópico-consulta%20%7C%20valores-teal)
![Dificuldade](https://img.shields.io/badge/dificuldade-Intermediário-yellow)
![Pré-req](https://img.shields.io/badge/pré--req-dataframe%20%7C%20filtros-purple)
![Biblioteca](https://img.shields.io/badge/requer-pip%20install%20pandas-orange)

> Além de filtrar, o pandas também permite **criar novas colunas** a partir de colunas já existentes e **consultar ou modificar valores específicos** dentro do DataFrame — usando os métodos `loc` e `iloc`.


## 📋 Conteúdo

1. [Preparando as bases de dados](#️-1-preparando-as-bases-de-dados)
2. [Criando novas colunas a partir de dados existentes](#-2-criando-novas-colunas-a-partir-de-dados-existentes)
3. [loc vs iloc: entendendo os dois métodos](#-3-loc-vs-iloc-entendendo-os-dois-métodos)
4. [Modificando um valor específico: desafio](#-4-modificando-um-valor-específico-desafio)

---

## ⚙️ 1. Preparando as bases de dados

- Preparando as bases de dados (o que fizemos nas últimas aulas)

In [4]:
import pandas as pd

# importando os arquivos
vendas_df = pd.read_csv(r'../spec/Contoso - Vendas - 2017.csv', sep=';', encoding='latin1')
produtos_df = pd.read_csv(r'../spec/Contoso - Produtos Cadastro Completo.csv', sep=';', encoding='latin1')
lojas_df = pd.read_csv(r'../spec/Contoso - Lojas.csv', sep=';', encoding='latin1')
clientes_df = pd.read_csv(r'../spec/Contoso - Clientes.csv', sep=';', encoding='latin1')

# Ajustar nome das colunas
produtos_df.rename(columns={'ÿNome do Produto': 'Nome do Produto'}, inplace=True)
clientes_df.rename(columns={'ÿID Cliente': 'ID Cliente'}, inplace=True)
lojas_df.rename(columns={'ÿID Loja': 'ID Loja'}, inplace=True)

# limpando apenas as colunas que queremos
clientes_df = clientes_df[['ID Cliente', 'E-mail']]
produtos_df = produtos_df[['ID Produto', 'Nome do Produto']]
lojas_df = lojas_df[['ID Loja', 'Nome da Loja']]

# mesclando e renomeando os dataframes
vendas_df = vendas_df.merge(produtos_df, on='ID Produto')
vendas_df = vendas_df.merge(lojas_df, on='ID Loja')
vendas_df = vendas_df.merge(clientes_df, on='ID Cliente').rename(columns={'E-mail': 'E-mail do Cliente'})

display(vendas_df.head())

,Numero da Venda,Data da Venda,Data do Envio,ID Canal,ID Loja,ID Produto,ID Promocao,ID Cliente,Quantidade Vendida,Quantidade Devolvida,Nome do Produto,Nome da Loja,E-mail do Cliente
0,1,01/01/2017,02/01/2017,1,86,981,2,6825,9,1,A. Datum Advanced Digital Camera M300 Pink,Loja Contoso Austin,rbrumfieldmy@ameblo.jp
1,2,01/01/2017,06/01/2017,5,308,1586,2,18469,9,1,SV DVD 55DVD Storage Binder M56 Black,Loja Contoso North America Reseller,cshawd4@technorati.com
2,3,01/01/2017,01/01/2017,0,294,1444,5,19730,13,1,"The Phone Company Touch Screen Phones 26-2.2"" ...",Loja Contoso Tehran No.2,kgorriekd@bandcamp.com
3,4,01/01/2017,01/01/2017,0,251,1468,5,29326,6,1,Contoso Touch Screen Phones - CRT M11 Black,Loja Contoso Singapore,angela49@adventure-works.com
4,5,01/01/2017,07/01/2017,6,94,1106,2,22617,4,1,Contoso SLR Camera M146 Orange,Loja Contoso Grand Prairie,jacob4@adventure-works.com


## 📅 2. Criando novas colunas a partir de dados existentes

Para criar uma nova coluna em um DataFrame, basta atribuir um valor (ou uma `Series`) a uma coluna que ainda não existe — o pandas cria ela automaticamente.

Vamos criar 3 novas colunas separando o dia, o mês e o ano da coluna `Data da Venda` (que hoje é só um texto no formato `dd/mm/aaaa`).

| Método | O que faz |
|--------|-----------|
| `dataframe['coluna'].str.split('/')` | Quebra o texto em uma lista, dividindo pelo caractere `/` |
| `.str[i]` | Pega o item de índice `i` dessa lista (0 = dia, 1 = mês, 2 = ano) |
| `dataframe['nova_coluna'] = valor` | Cria (ou substitui) uma coluna no DataFrame |

> 💡 **Dica:** se a coluna já existir, essa mesma atribuição serve para **modificar** todos os valores dela de uma vez.

In [5]:
vendas_df['Dia Venda'] = vendas_df['Data da Venda'].str.split('/').str[0]
vendas_df['Mes Venda'] = vendas_df['Data da Venda'].str.split('/').str[1]
vendas_df['Ano Venda'] = vendas_df['Data da Venda'].str.split('/').str[2]

display(vendas_df[['Data da Venda', 'Dia Venda', 'Mes Venda', 'Ano Venda']].head())

,Data da Venda,Dia Venda,Mes Venda,Ano Venda
0,01/01/2017,01,01,2017
1,01/01/2017,01,01,2017
2,01/01/2017,01,01,2017
3,01/01/2017,01,01,2017
4,01/01/2017,01,01,2017


## 🎯 3. loc vs iloc: entendendo os dois métodos

Para acessar (ou modificar) um valor específico do DataFrame, o pandas oferece dois métodos parecidos, mas com uma diferença importante:

| Método | Como funciona | Quando dá erro |
|--------|---------------|----------------|
| `loc` | Acessa pelo **rótulo** (label) do índice e da coluna | Se o rótulo não existir |
| `iloc` | Acessa pela **posição** (número da linha e da coluna, começando em 0) | Se a posição estiver fora do DataFrame |

> 💡 **Dica:** use `loc` quando o índice tiver um significado (como o nome do produto); use `iloc` quando só importa a posição, independente do rótulo.

Para praticar, vamos importar de novo a base de produtos — dessa vez com **todas** as colunas, incluindo o preço.

In [6]:
novo_produtos_df = pd.read_csv(r'../spec/Contoso - Produtos Cadastro Completo.csv', sep=';', encoding='latin1')
novo_produtos_df.rename(columns={'ÿNome do Produto': 'Nome do Produto'}, inplace=True)

display(novo_produtos_df.head())
# repare no .head() para pegar apenas os primeiros valores, é bem comum esse uso para ter uma visão do que são os dados

,Nome do Produto,Descricao do Produto,Fabricante,Nome da Marca,Tipo,Custo Unitario,Preco Unitario,ID Produto,ID Subcategoria
0,Contoso Wireless Laser Mouse E50 Grey,Advanced 2.4 GHz cordless technology makes fre...,"Contoso, Ltd",Contoso,Econômico,"10,69","20,96",873,22
1,Contoso Optical Wheel OEM PS/2 Mouse E60 Grey,"PS/2 mouse, 6 feet mouse cable","Contoso, Ltd",Contoso,Econômico,"6,63",13,879,22
2,Contoso Optical Wheel OEM PS/2 Mouse E60 Black,"PS/2 mouse, 6 feet mouse cable","Contoso, Ltd",Contoso,Econômico,"6,63",13,880,22
3,Contoso Optical Wheel OEM PS/2 Mouse E60 White,"PS/2 mouse, 6 feet mouse cable","Contoso, Ltd",Contoso,Econômico,"6,63",13,881,22
4,Contoso Optical Wheel OEM PS/2 Mouse E60 Silver,"PS/2 mouse, 6 feet mouse cable","Contoso, Ltd",Contoso,Econômico,"6,63",13,882,22


### 🔑 Definindo o índice pelo nome do produto

Para usar `loc` de um jeito mais natural, vamos definir a coluna `Nome do Produto` como índice do DataFrame, com `set_index`.

```python
novo_produtos_df = novo_produtos_df.set_index('Nome do Produto')
```

Agora vamos pegar o preço do produto **Contoso Optical Wheel OEM PS/2 Mouse E60 Black** dos dois jeitos: por `loc` (pelo nome) e por `iloc` (pela posição).

In [7]:
novo_produtos_df = novo_produtos_df.set_index('Nome do Produto')
display(novo_produtos_df.head())

,Descricao do Produto,Fabricante,Nome da Marca,Tipo,Custo Unitario,Preco Unitario,ID Produto,ID Subcategoria
Nome do Produto,,,,,,,,
Contoso Wireless Laser Mouse E50 Grey,Advanced 2.4 GHz cordless technology makes fre...,"Contoso, Ltd",Contoso,Econômico,"10,69","20,96",873,22
Contoso Optical Wheel OEM PS/2 Mouse E60 Grey,"PS/2 mouse, 6 feet mouse cable","Contoso, Ltd",Contoso,Econômico,"6,63",13,879,22
Contoso Optical Wheel OEM PS/2 Mouse E60 Black,"PS/2 mouse, 6 feet mouse cable","Contoso, Ltd",Contoso,Econômico,"6,63",13,880,22
Contoso Optical Wheel OEM PS/2 Mouse E60 White,"PS/2 mouse, 6 feet mouse cable","Contoso, Ltd",Contoso,Econômico,"6,63",13,881,22
Contoso Optical Wheel OEM PS/2 Mouse E60 Silver,"PS/2 mouse, 6 feet mouse cable","Contoso, Ltd",Contoso,Econômico,"6,63",13,882,22


In [8]:
# por loc (usando o nome do produto, já que ele é o índice)
preco_por_loc = novo_produtos_df.loc['Contoso Optical Wheel OEM PS/2 Mouse E60 Black', 'Preco Unitario']
print(f'Por loc: {preco_por_loc}')

# por iloc (usando a posição da linha e da coluna)
linha = novo_produtos_df.index.get_loc('Contoso Optical Wheel OEM PS/2 Mouse E60 Black')
coluna = novo_produtos_df.columns.get_loc('Preco Unitario')
preco_por_iloc = novo_produtos_df.iloc[linha, coluna]
print(f'Por iloc: {preco_por_iloc}')

Por loc: 13
Por iloc: 13


## 🔧 4. Modificando um valor específico: desafio

A empresa decidiu aumentar o preço do produto **ID 873** (`Contoso Wireless Laser Mouse E50 Grey`) para **23 reais**. Como fazemos essa alteração direto na base?

Assim como usamos `loc` para consultar um valor, também podemos usar para **atribuir** um novo valor à mesma posição:

```python
dataframe.loc[índice_linha, 'coluna'] = novo_valor
```

| Antes | Depois |
|-------|--------|
| `loc` só lê o valor | `loc` escreve o valor diretamente na base |

> ⚠️ **Cuidado:** essa alteração modifica o DataFrame na memória. Se quiser guardar o resultado num arquivo, é preciso exportar (`to_csv`, `to_excel`...) depois.

In [14]:
print(f'Preço antes: R${novo_produtos_df.loc["Contoso Wireless Laser Mouse E50 Grey", "Preco Unitario"]:.1f}')

novo_produtos_df.loc['Contoso Wireless Laser Mouse E50 Grey', 'Preco Unitario'] = 23

print(f'Preço depois: R${novo_produtos_df.loc["Contoso Wireless Laser Mouse E50 Grey", "Preco Unitario"]}')

Preço antes: R$23.0
Preço depois: R$23
